# GwenLand cuda-oxide Kernel Research -- glcuda Turing INT8 GEMM

Generates the `gl_gemm_mma_q8` / `gl_gemm_mma_q8_r256` INT8 tensor-core GEMM kernels via [cuda-oxide](https://github.com/NVlabs/cuda-oxide) (NVlabs, alpha) and benchmarks the resulting PTX against glcuda's hand-written PTX on a T4.

**cuda-oxide is used purely as a PTX *generator* here.** It is never added as a dependency of the GwenLand monorepo; `glcuda_sm75.ptx` is only modified if you explicitly copy a winning `.entry` block back in after Gate C review.

**Before you run this on real prefill-scratch shapes**: `gl_gemm_mma_q8_r256` has a known, unresolved `CUDA_ERROR_MISALIGNED_ADDRESS` bug when driven by the real engine (`glcuda/src/runner.rs:146-165`) that has never reproduced under this harness's synthetic buffers. A clean run here does **not** clear that bug -- it's a separate, real finding to carry back, not something this notebook resolves.

Research provenance: cuda-oxide pinned to commit [`a77daf85d1f4`](https://github.com/NVlabs/cuda-oxide/commit/a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c) (verified real, alpha, requires Rust nightly + CUDA 12.x+ + LLVM 21+ as of 2026-08-04). Every API used below (`#[cuda_module]`, `#[kernel]`, `DisjointSlice`, `SharedArray`, `thread::*`, `cuda_device::wmma::mma_m8n8k16_s32_s8`, `cudarc` 0.19.8) was checked against real upstream source at that commit -- cells marked **VERIFY** are the few spots that check stayed inconclusive; resolve those from source (Cell 3 dumps it) before trusting the generated code blind.

## 1. System deps: LLVM 21+

In [ ]:
%%bash
set -e
# cuda-oxide's docs: "we emit TMA/tcgen05/WGMMA intrinsics that llc
# from LLVM 20 and earlier can't handle" -- LLVM 21+ required even
# though OUR kernel only needs classic mma.sync (m8n8k16), because the
# pipeline as a whole (rustc-codegen-cuda -> Pliron IR -> LLVM IR ->
# llc) is built against llc-21/llc-22.
wget -qO- https://apt.llvm.org/llvm.sh | sudo bash -s -- 21
sudo apt-get install -y llvm-21-dev libpolly-21-dev clang-21
which llc-21 || echo 'llc-21 not on PATH -- add /usr/lib/llvm-21/bin manually'


## 2. Rust nightly + cargo-oxide

In [ ]:
%%bash
set -e
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none
source "$HOME/.cargo/env"
# README-documented toolchain as of research time; check
# https://github.com/NVlabs/cuda-oxide#installation for drift before
# assuming this pin is still current.
TOOLCHAIN=nightly-2026-04-03
rustup toolchain install "$TOOLCHAIN"
rustup component add rust-src rustc-dev llvm-tools --toolchain "$TOOLCHAIN"
cargo "+$TOOLCHAIN" install --git https://github.com/NVlabs/cuda-oxide.git cargo-oxide
cargo-oxide --version || echo 'cargo-oxide installed but --version failed -- check cargo oxide --help instead'
# --default-toolchain none above means plain `cargo`/`rustc` (no +toolchain)
# resolve to nothing until a default is set -- `cargo oxide ...` doesn't
# need this (it execs the standalone cargo-oxide binary directly, no
# rustup toolchain resolution involved), but runner/ (Section 9, plain
# cudarc, no nightly features) calls bare `cargo run` and would hit
# "error: no default toolchain configured" without this. Nightly is a
# superset of stable, so defaulting to the one toolchain we have anyway
# is fine for a crate that uses no nightly-only features.
rustup default "$TOOLCHAIN"


## 3. API discovery -- resolve the VERIFY markers from real source

Everything else in this notebook was checked against upstream source on GitHub during research. Two things were **not** independently confirmed and are load-bearing for `kernels/src/lib.rs` to compile:

1. Does `DisjointSlice::get_mut` accept a plain `usize`, or only index types built by `thread::index_1d()`/`index_2d_runtime()`?
2. Exact re-export path/name for `thread::blockDim_x()` (used `threadIdx_x`/`blockIdx_x` are confirmed from `tiled_gemm.rs`; `blockDim_x` was not exercised there).

This cell clones cuda-oxide at the pinned research commit and greps its own source for the real answer -- read the output before running Cell 5.

In [ ]:
%%bash
set -e
REV=a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c
if [ ! -d cuda-oxide ]; then git clone https://github.com/NVlabs/cuda-oxide.git; fi
cd cuda-oxide && git checkout "$REV" -q && cd ..
echo '--- DisjointSlice: full impl block ---'
grep -rn 'struct DisjointSlice\|impl.*DisjointSlice' cuda-oxide/crates/cuda-device/src/ | head -20
grep -rn -A5 'fn get_mut' cuda-oxide/crates/cuda-device/src/ | head -60
echo
echo '--- thread module: every pub fn (confirms threadIdx_x/blockDim_x/etc names) ---'
grep -rn 'pub fn ' cuda-oxide/crates/cuda-device/src/thread*.rs 2>/dev/null || \
  grep -rln 'mod thread' cuda-oxide/crates/cuda-device/src/ | xargs grep -n 'pub fn '
echo
echo '--- wmma module re-export of mma_m8n8k16_s32_s8 ---'
grep -rn 'mma_m8n8k16_s32_s8\|pub mod wmma\|pub use.*register_mma' cuda-oxide/crates/cuda-device/src/*.rs


**If `DisjointSlice::get_mut` turns out NOT to accept a raw `usize`:** the fix in `kernels/src/lib.rs` is either (a) construct the index via whatever type `get_mut` actually wants (check the grep output above for the real parameter type), or (b) replace the `DisjointSlice<f32>` output parameter with a raw `*mut f32` if cuda-oxide's `#[kernel]` macro accepts raw pointer parameters (grep `cuda-oxide/crates/cuda-macros/` for how it validates kernel signatures to check). Don't skip this -- it's the one part of the port most likely to not compile on the first try.

## 4. Materialize the workspace

In [ ]:
%%bash
# %%writefile does not create parent directories -- make them first,
# or every writefile cell below fails with FileNotFoundError.
mkdir -p gwenland-kernel-research/kernels/src
mkdir -p gwenland-kernel-research/runner/src


In [ ]:
%%writefile gwenland-kernel-research/Cargo.toml
# gwenland-kernel-research workspace
#
# NOTE: `kernels/` is deliberately NOT a member here. cuda-oxide's own
# examples mark device crates `[workspace]` (empty) so `cargo oxide build`
# invokes its special nightly + custom rustc-codegen-cuda backend without a
# parent workspace resolver getting in the way (see tiled_gemm's Cargo.toml
# upstream). Build kernels/ with `cargo oxide build` from inside kernels/,
# and runner/ with plain `cargo build` from here or inside runner/.
[workspace]
resolver = "3"
members = ["runner"]
exclude = ["kernels"]


In [ ]:
%%writefile gwenland-kernel-research/.gitignore
/target
/kernels/target
/runner/target
*.ptx.bak
Cargo.lock
reference/*.ptx


In [ ]:
%%writefile gwenland-kernel-research/kernels/Cargo.toml
[package]
name = "gwenland-kernels"
version = "0.1.0"
edition = "2024"

# Standalone crate, not a workspace member (see ../Cargo.toml comment and
# cuda-oxide's own examples/tiled_gemm/Cargo.toml, which does the same).
[workspace]

[dependencies]
# Pinned to the cuda-oxide commit verified during research (2026-08-04):
# https://github.com/NVlabs/cuda-oxide/commit/a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c
# Bump this rev deliberately, not by accident -- the project is alpha and
# its device-crate API has moved under active development.
cuda-device = { git = "https://github.com/NVlabs/cuda-oxide.git", rev = "a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c" }
cuda-host   = { git = "https://github.com/NVlabs/cuda-oxide.git", rev = "a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c" }
cuda-core   = { git = "https://github.com/NVlabs/cuda-oxide.git", rev = "a77daf85d1f42aac1ca363eb7b9b935d24ef2f3c" }

[lib]
crate-type = ["lib"]


In [ ]:
%%writefile gwenland-kernel-research/kernels/src/lib.rs
//! cuda-oxide equivalent of glcuda's Turing INT8 tensor-core GEMM kernels.
//!
//! Source of truth (hand-written PTX being compared against), verified
//! identical to this repo's working tree at research time:
//! https://raw.githubusercontent.com/gwenland-org/gwenland-ai/engine/gljax-bringup/glcuda/src/kernels/glcuda_sm75.ptx
//!
//! DESIGN NOTE: `gl_gemm_mma_q8` and `gl_gemm_mma_q8_r256` in that file are
//! the SAME algorithm at two m-tile counts (8 vs 32). `_r256` exists only
//! because writing a 32-way-unrolled body by hand needed a generator script
//! (`emit_r256.py`) -- in Rust that's just a runtime loop bound, so ONE
//! kernel below serves as the comparison target for both:
//!   - m_tiles = 8  -> compares against gl_gemm_mma_q8  (production kernel)
//!   - m_tiles = 32 -> compares against gl_gemm_mma_q8_r256 (has an
//!     unresolved CUDA_ERROR_MISALIGNED_ADDRESS bug in production driving --
//!     see glcuda/src/runner.rs:146-165. A clean cuda-oxide build that
//!     passes parity at m_tiles=32 would itself be a finding.)
//!
//! KNOWN GAP vs the hand kernel, by design choice, not oversight: `m_tiles`
//! is a runtime parameter here, not a const generic, so the accumulator
//! array is statically sized for MAX_M_TILES=32 regardless of the value
//! passed at launch. The hand kernel sizes its register bank exactly
//! (`.reg .f32 %f<32>` for the 8-tile version, `%f<80>` for r256) because
//! it's two separate hand-written bodies. This unified kernel likely loses
//! that register-pressure advantage in the m_tiles=8 case -- confirm via
//! `ptxas -v` register counts in the notebook's Task 3 cell. If the gap is
//! real, the fix is a `const M_TILES: u32` generic (monomorphize two kernel
//! instances) instead of a runtime parameter; that's the natural Task 5
//! follow-up, not implemented here to keep this first port simple enough to
//! debug against the parity test.
//!
//! VERIFY-ON-KAGGLE markers below are genuine unknowns, checked from real
//! source, not guessed. `cuda_device::wmma::mma_m8n8k16_s32_s8` was
//! confirmed to exist (lowers to exactly
//! `mma.sync.aligned.m8n8k16.row.col.s32.s8.s8.s32`, matching this kernel's
//! PTX target) by reading generated source on GitHub before this file was
//! first written.
//!
//! UPDATE (first real compile on a T4): `DisjointSlice::get_mut` does NOT
//! accept a raw `usize` -- it wants a `ThreadIndex<IndexSpace>`, only
//! mintable from `thread::index_1d()`/`index_2d()`, which give one
//! canonical index per thread. This kernel's per-thread writes (up to
//! `m_tiles*2` distinct (row,col) pairs from warp-fragment math, not a grid
//! position) don't fit that model. Fixed using the crate's own documented
//! escape hatch for this exact situation, `DisjointSlice::get_unchecked_mut`
//! (`crates/cuda-device/src/disjoint.rs:292`, doc example: "scatter
//! operations with known-unique destinations") -- see the epilogue below
//! for the uniqueness argument.
//!
//! That same investigation also surfaced two real correctness bugs in the
//! first draft, both fixed inline with comments where they were:
//! 1. The epilogue was dequantizing both of a lane's output columns with a
//!    single shared scale (`w_scales[w_row]`) instead of one scale per
//!    column (`w_scales[nc0]`, `w_scales[nc1]`) -- silently wrong numbers
//!    for one of every two output columns, invisible until a parity run
//!    hit a shape where those scales actually differ.
//! 2. Inactive warps (weight tile entirely past `out_dim`) had no
//!    equivalent of the hand PTX's `p11` predicate gating their store --
//!    their clamped-to-0 indices would race with the warp that legitimately
//!    owns column 0.
//! Neither was caught by the compiler; both were found by re-deriving what
//! the hand PTX's own comments actually say while chasing the ThreadIndex
//! fix, not by running anything. The parity check in `runner/` is what
//! would have caught them at runtime -- worth remembering next time
//! something merely compiles.

use cuda_device::{DisjointSlice, SharedArray, kernel, thread};
use cuda_host::cuda_module;

/// Rows staged into shared memory per staging pass. The hand kernel's
/// staging math (256 threads, 4 threads/row, 8 B/thread) covers 64 rows per
/// pass regardless of m_tiles -- gl_gemm_mma_q8_r256's comment says this
/// explicitly ("the k-loop staging runs FOUR passes to cover 0..255").
pub const STAGE_ROWS: usize = 64;

/// Upper bound this kernel supports (32 * 8 = 256 rows), matching
/// gl_gemm_mma_q8_r256's `ntok <= 256` contract. gl_gemm_mma_q8 uses
/// m_tiles = 8 and only exercises 1/4 of this budget.
pub const MAX_M_TILES: u32 = 32;

const SM_A_BYTES: usize = (MAX_M_TILES as usize / 8) * STAGE_ROWS * 32; // 8192
const SM_XS_FLOATS: usize = (MAX_M_TILES as usize / 8) * STAGE_ROWS; // 256

/// IEEE 754 binary16 bit pattern -> f32.
///
/// glcuda stores weight dequant scales as f16 (2 B/block: the PTX indexes
/// `wsc` with a `shl ..., 1`, i.e. a 2-byte stride). Converted by hand here
/// because native f16-in-device-code support in cuda-oxide is unverified as
/// of this port -- if `cuda_device` turns out to expose a hardware
/// `cvt.f32.f16`-backed type, prefer that over this software path (it's
/// pure ALU work the tensor-core kernel doesn't want on its critical path).
#[inline]
fn f16_bits_to_f32(bits: u16) -> f32 {
    let sign = (bits >> 15) as u32 & 1;
    let exp = (bits >> 10) as u32 & 0x1F;
    let frac = bits as u32 & 0x3FF;
    let out_bits = if exp == 0 {
        if frac == 0 {
            0u32
        } else {
            let mut e = -1i32;
            let mut m = frac;
            while m & 0x400 == 0 {
                m <<= 1;
                e -= 1;
            }
            m &= 0x3FF;
            ((e + 127 - 15) as u32) << 23 | (m << 13)
        }
    } else if exp == 0x1F {
        (0xFFu32) << 23 | (frac << 13)
    } else {
        (exp + 127 - 15) << 23 | (frac << 13)
    };
    f32::from_bits((sign << 31) | out_bits)
}

#[cuda_module]
mod kernels {
    use super::*;

    /// cuda-oxide equivalent of glcuda's `gl_gemm_mma_q8` / `gl_gemm_mma_q8_r256`.
    ///
    /// `Y[ntok,out] = X[ntok,in] @ W[out,in]^T` on Turing INT8 tensor cores,
    /// Q8_0-quantized weights + int8 activations, fused per-32-K dequant
    /// epilogue. `m_tiles` (8 or 32) selects which hand kernel this is being
    /// compared against.
    ///
    /// Launch: grid = ceil(out_dim / (8 * warps_per_block)), block = 256
    /// threads (8 warps/block, matches both hand kernels). Contract:
    /// `out_dim % 8 == 0`, `in_dim % 32 == 0`, `ntok <= m_tiles * 8`,
    /// `x_qs`/`x_scales` rows allocated up to `round8(ntok)`.
    ///
    /// Tolerance class: TOL_MATMUL (see architecture/ArchGLML_X2.md) --
    /// same class the hand kernel's parity test uses.
    #[kernel]
    pub fn gl_gemm_mma_q8_oxide(
        out_dim: u32,
        in_dim: u32,
        ntok: u32,
        m_tiles: u32,
        w_qs: &[i8],
        w_scales: &[u16],
        x_qs: &[i8],
        x_scales: &[f32],
        mut y: DisjointSlice<f32>,
    ) {
        static mut SM_A: SharedArray<i8, SM_A_BYTES> = SharedArray::UNINIT;
        static mut SM_XS: SharedArray<f32, SM_XS_FLOATS> = SharedArray::UNINIT;

        // CONFIRMED (first real compile on a T4, 2026-08-04): threadIdx_x,
        // blockDim_x, blockIdx_x, sync_threads, SharedArray, and the wmma
        // intrinsic below all compiled with zero errors -- the compiler's
        // only complaints were the two get_mut calls addressed above.
        let tid = thread::threadIdx_x();
        let ntid = thread::blockDim_x();
        let warp_id = tid >> 5;
        let lane = tid & 31;
        let warps_per_block = ntid >> 5;
        let ctaid = thread::blockIdx_x();
        let global_warp = ctaid * warps_per_block + warp_id;
        let n0 = global_warp * 8; // first weight row of this warp's output tile

        let group_id = (lane >> 2) as usize; // 0..8: output row within the 8x8 D tile
        let tig = (lane & 3) as usize; // 0..4: column-pair within the tile

        let out_dim_u = out_dim as usize;
        let in_dim_u = in_dim as usize;
        let nb = in_dim_u / 32; // number of 32-wide K blocks
        let ntok_pad8 = (ntok + 7) & !7u32;
        let stage_rows_total = m_tiles * 8; // 64 (m_tiles=8) or 256 (m_tiles=32)

        // Found alongside the get_unchecked_mut fix, by re-deriving what the
        // hand PTX's p11 predicate actually guards ("warps whose weight rows
        // fall outside `out` still stage and synchronize, and skip only
        // their own loads/MMAs/writes"): without this gate, an inactive
        // warp's clamped-to-0 nc0/nc1 (below) all alias idx (row*out_dim+0)
        // -- every inactive warp racing to stomp column 0 with garbage,
        // while the ONE real warp that owns column 0 is trying to write the
        // correct value there. The clamp-to-0 trick is only safe for LOADS
        // (reading garbage you then discard); it is a correctness bug for
        // STORES unless the whole warp additionally skips the store, which
        // is exactly what p11 is for.
        let warp_active = n0 < out_dim;

        // Clamp-to-0 instead of predicating every load, same trick the hand
        // kernel uses ("cheaper than predicating every load in the hot loop").
        let w_row = {
            let r = n0 as usize + group_id;
            if r < out_dim_u { r } else { 0 }
        };
        let nc0 = {
            let c = n0 as usize + 2 * tig;
            if c < out_dim_u { c } else { 0 }
        };
        let nc1 = {
            let c = nc0 + 1;
            if c < out_dim_u { c } else { 0 }
        };

        // One [f32; 2] accumulator per m-tile. Sized for MAX_M_TILES always
        // -- see the module doc's "known gap" note on why this likely costs
        // registers in the m_tiles=8 case that the hand kernel doesn't pay.
        let mut acc = [[0.0f32; 2]; MAX_M_TILES as usize];

        let mut kb = 0usize;
        while kb < nb {
            // --- Stage this K-block's 32 B/row activation slice + f32 scales ---
            let mut pass = 0u32;
            while pass * (STAGE_ROWS as u32) < stage_rows_total {
                let row_in_pass = tid / 4;
                let byte_off = (tid % 4) * 8;
                let row = pass * (STAGE_ROWS as u32) + row_in_pass;
                if row < ntok_pad8 {
                    let src = row as usize * in_dim_u + kb * 32 + byte_off as usize;
                    // BUG (found from real hardware output, not inspection):
                    // this used to be `row_in_pass * 32 + byte_off`, which
                    // never incorporates `pass` -- all 4 passes (m_tiles=32
                    // needs stage_rows_total=256, STAGE_ROWS=64) wrote to
                    // the SAME SM_A[0..2047] bytes, each overwriting the
                    // last, while SM_A[2048..8191] (where rows 64-255 are
                    // supposed to land) was never written at all. Silent
                    // for m_tiles=8 (only 1 pass, row==row_in_pass always)
                    // and for m_tiles=32 at small ntok (later passes' `row
                    // < ntok_pad8` gate suppressed their writes entirely,
                    // so pass 0's correct data was never disturbed and the
                    // epilogue never needed the corrupted region) -- only
                    // visible once ntok was large enough that all 4 passes
                    // actually ran (case 3, ntok=256): max_abs_diff=141.61,
                    // wrong for EVERY m-tile, not just m>=8, because pass 3
                    // clobbered pass 0's data too. `row` already has the
                    // pass offset folded in (computed right above for
                    // `src`) -- reuse it instead of the pass-blind
                    // `row_in_pass`.
                    let dst = (row as usize) * 32 + byte_off as usize;
                    unsafe {
                        let mut b = 0usize;
                        while b < 8 {
                            SM_A[dst + b] = x_qs[src + b];
                            b += 1;
                        }
                    }
                }
                pass += 1;
            }
            // Activation scales: stage_rows_total <= blockDim (256), so one
            // thread-per-row covers it in a single pass (matches the hand
            // kernel: "with 256 threads that is one row/thread").
            if tid < stage_rows_total {
                let row = tid;
                if row < ntok_pad8 {
                    unsafe {
                        SM_XS[row as usize] = x_scales[row as usize * nb + kb];
                    }
                }
            }
            thread::sync_threads();

            // Everything from here to the end of the k-block (B-fragment
            // load, both MMAs, and the epilogue accumulate) is gated on
            // warp_active -- see the comment on its declaration above.
            // Staging above this point stays unconditional; every warp keeps
            // participating in bar.sync regardless.
            if warp_active {
                // --- This warp's B fragment (weight bytes) for this K-block ---
                let w_base = w_row * in_dim_u + kb * 32;
                let lo = tig * 4;
                let hi = 16 + lo;
                let (b0, b1) = unsafe {
                    let b0 = u32::from_le_bytes([
                        w_qs[w_base + lo] as u8,
                        w_qs[w_base + lo + 1] as u8,
                        w_qs[w_base + lo + 2] as u8,
                        w_qs[w_base + lo + 3] as u8,
                    ]);
                    let b1 = u32::from_le_bytes([
                        w_qs[w_base + hi] as u8,
                        w_qs[w_base + hi + 1] as u8,
                        w_qs[w_base + hi + 2] as u8,
                        w_qs[w_base + hi + 3] as u8,
                    ]);
                    (b0, b1)
                };
                // BUG FOUND while chasing the get_mut/get_unchecked_mut
                // question below (not by inspection -- re-derived from the
                // hand PTX's own comment while tracing it): the scale that
                // dequantizes this lane's TWO accumulator columns is NOT
                // w_scales[w_row], it's one scale PER OUTPUT COLUMN --
                // w_scales[nc0] for acc[.][0] and w_scales[nc1] for
                // acc[.][1]. w_row (=n0+group_id) is a completely different
                // row than nc0/nc1 (=n0+2*tig, +1) in general -- it's the
                // row THIS lane loads for the B fragment, which the m8n8k16
                // ISA fragment layout deliberately decouples from which
                // columns this lane's D fragment accumulates. The hand PTX
                // has two separate "epilogue scale walkers" (nc0, nc1) for
                // exactly this reason; a single `wsc` shared across both
                // columns (what an earlier version of this file did)
                // silently dequantizes with the wrong scale for one of every
                // two output columns whenever nc0/nc1's weight rows don't
                // happen to share a Q8_0 scale with w_row. Parity would have
                // caught this the moment TOL_MATMUL got exercised on a case
                // where row scales actually differ -- catching it here first
                // instead.
                let wsc0 = f16_bits_to_f32(w_scales[nc0 * nb + kb]);
                let wsc1 = f16_bits_to_f32(w_scales[nc1 * nb + kb]);

                // --- Chain two m8n8k16 MMAs (K=16 each) across this warp's m-tiles ---
                let mut m = 0u32;
                while m < m_tiles {
                    let row_base = m * 8;
                    if row_base < stage_rows_total {
                        let a_row = row_base as usize + group_id;
                        let a_base = a_row * 32;
                        let (a0, a1) = unsafe {
                            let a0 = u32::from_le_bytes([
                                SM_A[a_base + lo] as u8,
                                SM_A[a_base + lo + 1] as u8,
                                SM_A[a_base + lo + 2] as u8,
                                SM_A[a_base + lo + 3] as u8,
                            ]);
                            let a1 = u32::from_le_bytes([
                                SM_A[a_base + hi] as u8,
                                SM_A[a_base + hi + 1] as u8,
                                SM_A[a_base + hi + 2] as u8,
                                SM_A[a_base + hi + 3] as u8,
                            ]);
                            (a0, a1)
                        };

                        // CONFIRMED both by source read and by a real
                        // compile: `cuda_device::wmma::mma_m8n8k16_s32_s8`.
                        let mut c = [0i32; 2];
                        c = unsafe { cuda_device::wmma::mma_m8n8k16_s32_s8(c, a0, b0) };
                        c = unsafe { cuda_device::wmma::mma_m8n8k16_s32_s8(c, a1, b1) };

                        let xsc = unsafe { SM_XS[a_row] };
                        acc[m as usize][0] += c[0] as f32 * wsc0 * xsc;
                        acc[m as usize][1] += c[1] as f32 * wsc1 * xsc;
                    }
                    m += 1;
                }
            }
            thread::sync_threads(); // before the next K-block restages SM_A/SM_XS
            kb += 1;
        }

        // --- Epilogue: up to m_tiles rows x 2 columns each, predicated on
        // ntok AND warp_active (see the comment on warp_active's
        // declaration above -- without this an inactive warp's clamped
        // nc0/nc1 alias column 0 and race with the real owner of column 0).
        if warp_active {
            let mut m = 0u32;
            while m < m_tiles {
                let row = m * 8 + group_id as u32;
                if row < ntok {
                    let idx0 = row as usize * out_dim_u + nc0;
                    let idx1 = row as usize * out_dim_u + nc1;
                    // RESOLVED (was a VERIFY marker): DisjointSlice::get_mut
                    // takes a `ThreadIndex<IndexSpace>`, not a raw usize --
                    // confirmed from the real compiler error against
                    // crates/cuda-device/src/disjoint.rs:245 at the pinned rev.
                    // ThreadIndex is only mintable from thread::index_1d() /
                    // index_2d(), which give ONE canonical index per thread --
                    // this kernel's per-thread writes (up to m_tiles*2 distinct
                    // (row,col) pairs, computed from warp-fragment math, not a
                    // grid position) don't fit that model at all. The crate's
                    // own escape hatch is exactly this situation --
                    // `get_unchecked_mut`'s doc example is literally "scatter
                    // operations with known-unique destinations".
                    //
                    // SAFETY: idx0 != idx1 (nc1 = nc0 + 1, and out_dim % 8 == 0
                    // by contract so the row never straddles a boundary between
                    // them). Across lanes: nc0/nc1 depend on tig = lane % 4, so
                    // the 4 lanes sharing a group_id touch 8 disjoint columns
                    // between them; across group_id (0..8) the row differs. No
                    // two (m, lane) pairs land on the same (row, col) -- same
                    // uniqueness argument the hand PTX relies on for its
                    // unpredicated `st.global.f32`. warp_active (checked
                    // above) means nc0/nc1 never actually need their clamp;
                    // `row < ntok` keeps the row in-bounds too, matching
                    // get_unchecked_mut's contract.
                    unsafe {
                        *y.get_unchecked_mut(idx0) = acc[m as usize][0];
                        *y.get_unchecked_mut(idx1) = acc[m as usize][1];
                    }
                }
                m += 1;
            }
        }
    }
}


In [ ]:
%%writefile gwenland-kernel-research/kernels/src/main.rs
//! Every `cargo oxide` example I found upstream (tiled_gemm, wgmma, ...) is a
//! single-file bin crate with host code and `#[cuda_module]` device code
//! together. I could not confirm from outside whether `cargo oxide build`
//! extracts PTX from a pure-lib crate. This binary exists only so the
//! package has a bin target either way -- if Cell 5 of the notebook shows
//! `cargo oxide build` works fine against `--lib`, delete this file.
fn main() {
    println!("gwenland-kernels: device code lives in lib.rs; this bin only exists");
    println!("to guarantee cargo-oxide has a target to extract PTX from.");
}


In [ ]:
%%writefile gwenland-kernel-research/runner/Cargo.toml
[package]
name = "gwenland-kernel-runner"
version = "0.1.0"
edition = "2021"

# runner/ is plain stable-toolchain-compatible Rust -- it only LOADS and
# LAUNCHES already-compiled .ptx files via cudarc's driver API. It never
# compiles device code itself, so it does not need cuda-oxide's nightly
# toolchain. This is deliberate: cuda-oxide is a PTX *generator* here, and
# the benchmark harness stays uniform across "hand-written PTX" and
# "cuda-oxide-generated PTX" -- both are just PTX files to cudarc.
[dependencies]
cudarc = { version = "0.19.8", default-features = false, features = [
    "driver",
    "cuda-version-from-build-system",
    # default-features=false drops "fallback-dynamic-loading" too (it's a
    # DEFAULT feature, not something on regardless) -- cudarc's build.rs
    # panics with "None between dynamic-loading, fallback-dynamic-loading,
    # dynamic-linking and static-linking features are active, this is a bug"
    # without one of these explicitly re-added. fallback-dynamic-loading
    # (dlopen libcuda.so at runtime, falls back gracefully if absent) is the
    # right choice for a Kaggle T4 instance where we don't control how CUDA
    # was installed.
    "fallback-dynamic-loading",
    # `Ptx` (from_file/from_src) AND `CudaContext::load_module` are BOTH
    # `#[cfg(feature = "nvrtc")]`-gated in cudarc's source (driver/safe/
    # core.rs:2173) even though we never call nvrtc's actual runtime
    # compiler (compile_ptx) -- the feature gates the whole `nvrtc` module,
    # Ptx type included, not just the compilation entry points. Confirmed
    # by reading cudarc v0.19.8 source directly, not from docs.
    "nvrtc",
] }
rand = "0.8"


In [ ]:
%%writefile gwenland-kernel-research/runner/src/main.rs
//! Ceiling Sprint Phase 3 harness — Changes 1A + 1B.
//!
//! CHANGE 1A (re-baseline). Every variant is now measured as
//! `1 warmup launch -> correctness check -> ITERS launches -> ONE sync`,
//! matching `glcuda/examples/bench.rs`'s existing `[mma gate]` pattern.
//! The previous version timed a single COLD launch with a `synchronize()`
//! inside the timed region, which measured launch overhead, not the kernel:
//! Phase 1 showed case 1 and case 2 differ by 224x in work but only 1.9x in
//! measured time, which is impossible for a kernel-bound measurement.
//!
//! CHANGE 1B (2D grid). The hand kernel is now measured on BOTH paths:
//!   * `[1D chunked]`  original PTX + the host-side 64-row chunk loop that
//!                     `glcuda/src/runner.rs:166-180` actually runs in
//!                     production. This is the honest baseline -- it
//!                     includes the chunking cost Phase 1 quantified
//!                     (intensity 273 -> 101 ops/byte).
//!   * `[2D grid]`     patched PTX, ONE launch, grid.y = ceil(ntok/64).
//! Loading both the original and the patched module (rather than reusing the
//! patched one with grid.y=1 for the baseline) means the A/B also catches any
//! cost the patch itself introduces.
//!
//! `gl_gemm_mma_q8_r256` is measured for CORRECTNESS ONLY -- Phase 1 finding
//! 4 established it computes wrong numbers on aligned synthetic buffers, and
//! timing a broken kernel is meaningless.
//!
//! Reminder for reading the output: a win here is DIAGNOSTIC ONLY. Per Phase
//! 2, the merge gate is glbench `prefill_tps` on a real model. This repo has
//! twice recorded a ~2x isolated win that was neutral in production
//! (VNNI-512, row-tile GEMM).

use cudarc::driver::{
    CudaContext, CudaFunction, CudaSlice, CudaStream, DevicePtr, DevicePtrMut, LaunchConfig,
    PushKernelArg,
};
use cudarc::nvrtc::Ptx;
use std::path::PathBuf;
// cudarc's allocation and host<->device copy methods (`alloc_zeros`,
// `clone_htod`, `clone_dtoh`) take `self: &Arc<Self>`, NOT `&self`
// (driver/safe/core.rs:1559-1631), so every helper that copies has to hold
// the Arc rather than a plain &CudaStream. `launch_builder`, `synchronize`
// and `device_ptr` are ordinary `&self` methods and deref-coerce from the
// Arc for free.
use std::sync::Arc;
use std::time::{Duration, Instant};

/// Tolerance for the INT8 tensor-core GEMM vs the scalar dequant reference.
/// Wide because the s32 accumulate is exact but the f16 weight-scale round
/// trip is not; the sprint brief fixes this figure at 5.0e-2.
const TOL_MATMUL: f32 = 5.0e-2;

/// Measured iterations per variant (matches bench.rs's `[mma gate]`).
const ITERS: u32 = 50;

/// T4 spec figures, for reporting only.
const T4_SMS: u32 = 40;
const T4_INT8_TOPS: f64 = 65.0;

struct Q8Case {
    out_dim: usize,
    in_dim: usize,
    ntok: usize,
}

fn quantize_q8_0(rows: usize, cols: usize, data: &[f32]) -> (Vec<i8>, Vec<f32>, usize) {
    let nb = cols / 32;
    let mut qs = vec![0i8; rows * cols];
    let mut scales = vec![0f32; rows * nb];
    for r in 0..rows {
        for b in 0..nb {
            let base = r * cols + b * 32;
            let block = &data[base..base + 32];
            let amax = block.iter().fold(0f32, |m, &v| m.max(v.abs()));
            let scale = if amax > 0.0 { amax / 127.0 } else { 1.0 };
            scales[r * nb + b] = scale;
            for i in 0..32 {
                qs[base + i] = (block[i] / scale).round().clamp(-127.0, 127.0) as i8;
            }
        }
    }
    (qs, scales, nb)
}

fn scalar_reference(
    w_qs: &[i8],
    w_scales: &[f32],
    x_qs: &[i8],
    x_scales: &[f32],
    out_dim: usize,
    in_dim: usize,
    ntok: usize,
) -> Vec<f32> {
    let nb = in_dim / 32;
    let mut y = vec![0f32; ntok * out_dim];
    for t in 0..ntok {
        for o in 0..out_dim {
            let mut acc = 0f32;
            for b in 0..nb {
                let wsc = w_scales[o * nb + b];
                let xsc = x_scales[t * nb + b];
                let mut dot = 0i32;
                for k in 0..32 {
                    dot += w_qs[o * in_dim + b * 32 + k] as i32
                        * x_qs[t * in_dim + b * 32 + k] as i32;
                }
                acc += dot as f32 * wsc * xsc;
            }
            y[t * out_dim + o] = acc;
        }
    }
    y
}

fn f32_to_f16_bits(v: f32) -> u16 {
    let bits = v.to_bits();
    let sign = (bits >> 16) & 0x8000;
    let exp = ((bits >> 23) & 0xFF) as i32 - 127 + 15;
    let frac = bits & 0x7FFFFF;
    let half = if exp <= 0 {
        0
    } else if exp >= 0x1F {
        0x7C00
    } else {
        ((exp as u32) << 10) | (frac >> 13)
    };
    (sign | half) as u16
}

fn max_abs_diff(a: &[f32], b: &[f32]) -> f32 {
    a.iter()
        .zip(b.iter())
        .map(|(x, y)| (x - y).abs())
        .fold(0f32, f32::max)
}

// ---------------------------------------------------------------------------
// Launch paths
// ---------------------------------------------------------------------------

/// Production-equivalent baseline: 1-D grid, host-side 64-row chunk loop.
/// Mirrors `glcuda/src/runner.rs:166-180` exactly, including the fact that
/// each chunk re-reads the entire weight matrix.
#[allow(clippy::too_many_arguments)]
fn launch_hand_1d_chunked(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    gx: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) {
    let nb = in_dim / 32;
    let cfg = LaunchConfig {
        grid_dim: (gx, 1, 1),
        block_dim: (256, 1, 1),
        shared_mem_bytes: 0,
    };
    let mut t0 = 0u32;
    while t0 < ntok {
        let nn = (ntok - t0).min(64);
        let xq = d_xqs.slice((t0 * in_dim) as usize..);
        let xs = d_xsc.slice((t0 * nb) as usize..);
        let mut yv = d_y.slice_mut((t0 * out_dim) as usize..);
        let mut b = stream.launch_builder(f);
        b.arg(d_wqs);
        b.arg(d_wsc);
        b.arg(&xq);
        b.arg(&xs);
        b.arg(&mut yv);
        b.arg(&out_dim);
        b.arg(&in_dim);
        b.arg(&nn);
        // Binding silences Option's #[must_use]; it is None unless
    // record_kernel_launch() requested built-in event timing (we time on the
    // host, around the whole 50-iteration loop).
    let _evt = unsafe { b.launch(cfg) }.unwrap();
        t0 += nn;
    }
}

/// Single launch with an EXPLICIT grid.
///
/// `gy` is a parameter, not derived, because two different callers need
/// different values and getting it wrong is silent:
///   * Change 1B candidate (patched kernel): `gy = ceil(ntok/64)`.
///   * `gl_gemm_mma_q8_r256` (NOT patched): `gy = 1` is mandatory. It has no
///     `%ctaid.y` read, so every block would ignore its y index and
///     recompute the same rows into the same output -- duplicate work and
///     racing writes, not a crash. Deriving `gy` inside this function once
///     did exactly that to r256.
#[allow(clippy::too_many_arguments)]
fn launch_hand_single(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    gx: u32,
    gy: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) {
    let cfg = LaunchConfig {
        grid_dim: (gx, gy.max(1), 1),
        block_dim: (256, 1, 1),
        shared_mem_bytes: 0,
    };
    let mut b = stream.launch_builder(f);
    b.arg(d_wqs);
    b.arg(d_wsc);
    b.arg(d_xqs);
    b.arg(d_xsc);
    b.arg(d_y);
    b.arg(&out_dim);
    b.arg(&in_dim);
    b.arg(&ntok);
    // Binding silences Option's #[must_use]; it is None unless
    // record_kernel_launch() requested built-in event timing (we time on the
    // host, around the whole 50-iteration loop).
    let _evt = unsafe { b.launch(cfg) }.unwrap();
}

/// cuda-oxide kernel. Needs an explicit (ptr, len) pair per slice: cudarc's
/// `.arg(&CudaSlice)` pushes only the pointer, but cuda-oxide compiles Rust
/// `&[T]` to a fat pointer, so pointer-only args desync every later
/// parameter (this was a real segfault earlier in this research pass).
#[allow(clippy::too_many_arguments)]
fn launch_oxide(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    cfg: LaunchConfig,
    m_tiles: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) -> Result<(), cudarc::driver::DriverError> {
    let wqs_len = d_wqs.len() as u64;
    let wsc_len = d_wsc.len() as u64;
    let xqs_len = d_xqs.len() as u64;
    let xsc_len = d_xsc.len() as u64;
    let y_len = d_y.len() as u64;
    let (wqs_ptr, _g1) = d_wqs.device_ptr(stream);
    let (wsc_ptr, _g2) = d_wsc.device_ptr(stream);
    let (xqs_ptr, _g3) = d_xqs.device_ptr(stream);
    let (xsc_ptr, _g4) = d_xsc.device_ptr(stream);
    let (y_ptr, _g5) = d_y.device_ptr_mut(stream);
    let mut b = stream.launch_builder(f);
    b.arg(&out_dim);
    b.arg(&in_dim);
    b.arg(&ntok);
    b.arg(&m_tiles);
    b.arg(&wqs_ptr);
    b.arg(&wqs_len);
    b.arg(&wsc_ptr);
    b.arg(&wsc_len);
    b.arg(&xqs_ptr);
    b.arg(&xqs_len);
    b.arg(&xsc_ptr);
    b.arg(&xsc_len);
    b.arg(&y_ptr);
    b.arg(&y_len);
    // `launch` returns Result<Option<(CudaEvent, CudaEvent)>, _> -- the
    // Option is Some only when `record_kernel_launch()` asked for built-in
    // event timing, which this harness does not (it times on the host
    // around the whole 50-iteration loop). Discard the None.
    unsafe { b.launch(cfg) }.map(|_| ())
}

// ---------------------------------------------------------------------------
// Measurement (Change 1A)
// ---------------------------------------------------------------------------

/// warmup -> correctness -> ITERS timed launches -> one sync.
/// The correctness check runs ONCE, on the warmup result, so the timed loop
/// measures only kernel time (no device-to-host copies inside it).
fn measure(
    stream: &Arc<CudaStream>,
    d_y: &mut CudaSlice<f32>,
    reference: &[f32],
    launch: &mut dyn FnMut(&mut CudaSlice<f32>),
) -> (Duration, f32) {
    launch(d_y);
    stream.synchronize().unwrap();
    let y_host = stream.clone_dtoh(d_y).unwrap();
    let diff = max_abs_diff(&y_host, reference);

    let t0 = Instant::now();
    for _ in 0..ITERS {
        launch(d_y);
    }
    stream.synchronize().unwrap();
    (t0.elapsed() / ITERS, diff)
}

#[allow(clippy::too_many_arguments)]
fn report(label: &str, dt: Duration, diff: f32, case: &Q8Case, blocks: u32, baseline: Option<Duration>) {
    let ops = 2.0 * case.out_dim as f64 * case.in_dim as f64 * case.ntok as f64;
    let tops = ops / dt.as_secs_f64() / 1e12;
    let status = if diff < TOL_MATMUL { "PASS" } else { "FAIL" };
    let sms = blocks.min(T4_SMS);
    let delta = match baseline {
        Some(b) if b.as_secs_f64() > 0.0 => format!(
            "  delta {:+.1}%",
            (b.as_secs_f64() / dt.as_secs_f64() - 1.0) * 100.0
        ),
        _ => String::new(),
    };
    println!(
        "  {label:<34} {:>8.2}us  {:>6.3} TOPS ({:>4.1}% of 65)  blocks={blocks:<3} ({sms}/{T4_SMS} SM)  max_abs_diff={diff:.3e} [{status}]{delta}",
        dt.as_secs_f64() * 1e6,
        tops,
        tops / T4_INT8_TOPS * 100.0,
    );
}

fn main() {
    let hand_ptx = std::env::var("HAND_PTX")
        .unwrap_or_else(|_| "reference/glcuda_sm75.ptx".to_string());
    let hand_ptx_orig = std::env::var("HAND_PTX_ORIG")
        .unwrap_or_else(|_| "reference/glcuda_sm75.ptx.bak".to_string());
    let oxide_ptx = std::env::var("OXIDE_PTX")
        .unwrap_or_else(|_| "../kernels/gwenland-kernels.ptx".to_string());

    let ctx = CudaContext::new(0).unwrap();
    let stream = ctx.default_stream();

    let m_orig = ctx
        .load_module(Ptx::from_file(PathBuf::from(&hand_ptx_orig)))
        .unwrap();
    let m_2d = ctx
        .load_module(Ptx::from_file(PathBuf::from(&hand_ptx)))
        .unwrap();
    let m_oxide = ctx
        .load_module(Ptx::from_file(PathBuf::from(&oxide_ptx)))
        .unwrap();

    let f_1d = m_orig.load_function("gl_gemm_mma_q8").unwrap();
    let f_r256 = m_orig.load_function("gl_gemm_mma_q8_r256").unwrap();
    let f_2d = m_2d.load_function("gl_gemm_mma_q8").unwrap();
    let f_oxide = m_oxide.load_function("gl_gemm_mma_q8_oxide").unwrap();

    println!("=== CEILING SPRINT -- GATE 3 RESULTS ===");
    println!("Hardware: T4, Kaggle");
    println!(
        "Methodology: 1 warmup launch + {ITERS} measured iterations, single sync after loop"
    );
    println!("Baseline path: original PTX + host 64-row chunk loop (== production runner.rs)");
    println!("Candidate path: patched PTX, single launch, grid.y = ceil(ntok/64)");

    // dim 896 (Qwen2.5-0.5B) is deliberately NOT a power of two; ntok=200 is
    // deliberately NOT a multiple of 64, to exercise the grid.y tail block.
    let cases = [
        Q8Case { out_dim: 64, in_dim: 128, ntok: 8 },
        Q8Case { out_dim: 256, in_dim: 896, ntok: 64 },
        Q8Case { out_dim: 512, in_dim: 4096, ntok: 256 },
        Q8Case { out_dim: 256, in_dim: 896, ntok: 200 },
    ];

    for case in &cases {
        println!(
            "\n=== case out={} in={} ntok={} ===",
            case.out_dim, case.in_dim, case.ntok
        );
        let ntok_pad8 = (case.ntok + 7) & !7;
        let out_u32 = case.out_dim as u32;
        let in_u32 = case.in_dim as u32;
        let ntok_u32 = case.ntok as u32;
        let gx = out_u32.div_ceil(64).max(1);
        let gy = ntok_u32.div_ceil(64).max(1);

        let mut rng_state = 0x2026_08_04u64;
        let mut next = || {
            rng_state ^= rng_state << 13;
            rng_state ^= rng_state >> 7;
            rng_state ^= rng_state << 17;
            ((rng_state % 2000) as f32 / 1000.0) - 1.0
        };
        let w_f32: Vec<f32> = (0..case.out_dim * case.in_dim).map(|_| next()).collect();
        let x_f32: Vec<f32> = (0..ntok_pad8 * case.in_dim).map(|_| next()).collect();

        let (w_qs, w_scales_f32, _) = quantize_q8_0(case.out_dim, case.in_dim, &w_f32);
        let (x_qs, x_scales, _) = quantize_q8_0(ntok_pad8, case.in_dim, &x_f32);
        let w_scales_f16: Vec<u16> = w_scales_f32.iter().map(|&s| f32_to_f16_bits(s)).collect();
        let reference = scalar_reference(
            &w_qs, &w_scales_f32, &x_qs, &x_scales, case.out_dim, case.in_dim, case.ntok,
        );

        let d_wqs = stream.clone_htod(&w_qs).unwrap();
        let d_wsc16 = stream.clone_htod(&w_scales_f16).unwrap();
        let d_xqs = stream.clone_htod(&x_qs).unwrap();
        let d_xsc = stream.clone_htod(&x_scales).unwrap();
        let mut d_y = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();

        // --- baseline: 1D grid + host chunk loop (production-equivalent) ---
        let (dt_1d, diff_1d) = {
            let mut f = |y: &mut CudaSlice<f32>| {
                launch_hand_1d_chunked(
                    &stream, &f_1d, gx, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y, out_u32, in_u32,
                    ntok_u32,
                )
            };
            measure(&stream, &mut d_y, &reference, &mut f)
        };
        report("[1D chunked] gl_gemm_mma_q8", dt_1d, diff_1d, case, gx, None);

        // --- candidate: 2D grid, single launch ---
        let (dt_2d, diff_2d) = {
            let mut f = |y: &mut CudaSlice<f32>| {
                launch_hand_single(
                    &stream, &f_2d, gx, gy, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y, out_u32, in_u32,
                    ntok_u32,
                )
            };
            measure(&stream, &mut d_y, &reference, &mut f)
        };
        report("[2D grid]    gl_gemm_mma_q8", dt_2d, diff_2d, case, gx * gy, Some(dt_1d));

        if gy == 1 {
            println!("      (grid.y=1 -> ctaid.y offset is a no-op; neutral is the EXPECTED result)");
        }
        if (diff_1d - diff_2d).abs() > 0.0 {
            println!(
                "      NOTE: 1D and 2D max_abs_diff differ ({diff_1d:.6e} vs {diff_2d:.6e}) -- \
                 for grid.y=1 cases they should be byte-identical; investigate before trusting timings"
            );
        }

        // --- r256: correctness only (Phase 1 finding 4) ---
        if case.ntok <= 256 {
            let mut yv = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();
            // gy = 1 is REQUIRED: r256 was deliberately not patched, so it has
            // no %ctaid.y read and covers all its rows (up to 256) from a 1-D
            // grid on its own.
            launch_hand_single(
                &stream, &f_r256, gx, 1, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, &mut yv, out_u32,
                in_u32, ntok_u32,
            );
            stream.synchronize().unwrap();
            let y_host = stream.clone_dtoh(&yv).unwrap();
            let d = max_abs_diff(&y_host, &reference);
            let st = if d < TOL_MATMUL { "PASS" } else { "FAIL" };
            println!(
                "  {:<34} correctness only, not timed          max_abs_diff={d:.3e} [{st}]",
                "[1D]         gl_gemm_mma_q8_r256"
            );
        }

        // --- cuda-oxide, 1-D grid (unchanged; diagnostic context only) ---
        for &m_tiles in &[8u32, 32u32] {
            if case.ntok > (m_tiles * 8) as usize {
                continue;
            }
            let cfg = LaunchConfig {
                grid_dim: (gx, 1, 1),
                block_dim: (256, 1, 1),
                shared_mem_bytes: 0,
            };
            let mut yv = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();
            let probe = launch_oxide(
                &stream, &f_oxide, cfg, m_tiles, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, &mut yv,
                out_u32, in_u32, ntok_u32,
            );
            if let Err(e) = probe {
                println!("  [oxide m{m_tiles}] LAUNCH FAILED: {e:?}");
                continue;
            }
            let (dt, diff) = {
                let mut f = |y: &mut CudaSlice<f32>| {
                    launch_oxide(
                        &stream, &f_oxide, cfg, m_tiles, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y,
                        out_u32, in_u32, ntok_u32,
                    )
                    .unwrap()
                };
                measure(&stream, &mut yv, &reference, &mut f)
            };
            report(
                &format!("[1D oxide]   m_tiles={m_tiles}"),
                dt,
                diff,
                case,
                gx,
                Some(dt_1d),
            );
        }
    }

    println!("\n--- REMINDER ---");
    println!("These are DIAGNOSTIC numbers. Per Phase 2 the merge gate is glbench");
    println!("prefill_tps on a real model; an isolated win here does not justify");
    println!("touching the monorepo PTX (VNNI-512 and row-tile both won isolated");
    println!("and went neutral in production).");
}


In [ ]:
%%writefile gwenland-kernel-research/patch_ptx_2d_grid.py
#!/usr/bin/env python3
"""
Ceiling Sprint Change 1B: add a ctaid.y token-row offset to gl_gemm_mma_q8.

Operates on the RESEARCH COPY of glcuda_sm75.ptx only. The monorepo PTX is
untouched until Gate 3 confirms a win.

WHAT THE PATCH DOES
-------------------
Today the grid is 1-D: grid = (ceil(out_dim/64), 1, 1). ntok never reaches the
grid at all -- runner.rs walks token rows on the HOST, launching the kernel
once per 64-row chunk with pre-offset pointers. Phase 1 measured the cost:
case 3 (out=512 in=4096 ntok=256) runs 4 sequential launches over 8 blocks =
8 of 40 SMs busy, and each chunk re-reads the whole 2.2 MB weight matrix from
DRAM (arithmetic intensity 273 -> 101 ops/byte, which flips the only
compute-bound shape in the suite to memory-bound).

This patch moves that chunk loop off the host and into the grid:

    grid = (ceil(out_dim/64), ceil(ntok/64), 1)

Each block rebases its three token-indexed pointers by t0 = ctaid.y * 64 and
clamps ntok to its own row count -- reproducing EXACTLY what the host loop
did per chunk. That is the whole reason this patch is safe: every instruction
after the insertion point still sees "a block with `ntok` rows starting at
x_qs", so the MMA math, the m8n8k16 fragment layout, the per-32-K scale
epilogue and the register accumulators are all bit-for-bit unchanged.

The 4 chunks now run CONCURRENTLY over one 2.2 MB weight matrix that fits in
the T4's 4 MB L2, so the redundant weight reads become L2 hits rather than
DRAM re-reads.

PADDING CONTRACT (verified, does not widen)
-------------------------------------------
The kernel reads staging rows up to round8(ntok). t0 is a multiple of 64 and
therefore of 8, so t0 + round8(nn) == round8(min(ntok, t0+64)) <= round8(ntok).
The existing "x_qs/x_scales allocated to round8(ntok)" contract is exactly
sufficient. Worked example, ntok=100: grid.y=2; block y=1 has t0=64, nn=36,
round8(36)=40, so it reads rows 64..104 -- and round8(100) is 104. Exact fit.

NOT PATCHED: gl_gemm_mma_q8_r256
--------------------------------
Phase 1 finding 4: r256 fails parity on all three shapes with synthetic
ALIGNED buffers (errors 15.74 / 54.375 / 136.05, growing with size), which is
a numerical bug distinct from its known CUDA_ERROR_MISALIGNED_ADDRESS crash.
Patching a kernel that computes wrong numbers would produce meaningless
timings. It is reported, not optimized.

REGISTER NAMING
---------------
New registers are NAMED (%r_gy_*, %rd_gy_*) rather than numbered. The file's
existing style is numbered (%r1..%r47, %rd1..%rd43) and this kernel is close
to its declared ceiling, so adding numbered registers would mean hand-counting
free slots -- exactly the "duplicate declaration in one function body" failure
ptx-writing.md rules 2-3 warn about (it cost a debugging session once). Named
registers cannot collide, and they are greppable.

ANCHOR SCOPING (the thing that makes this script safe)
------------------------------------------------------
Both anchors appear TWICE in the file -- once in gl_gemm_mma_q8 and once in
gl_gemm_mma_q8_r256 (verified: lines 75/495 and 86/506). A naive
str.replace() would corrupt r256. This script slices the file at the r256
entry line and patches only the region before it, asserting exactly one match
for each anchor. Any deviation aborts without writing.
"""
import sys

# Anchors, verified against glcuda_sm75.ptx at the sprint's baseline commit.
ENTRY_TARGET = ".visible .entry gl_gemm_mma_q8("
ENTRY_R256 = ".visible .entry gl_gemm_mma_q8_r256("
ANCHOR_REGS = "    .reg .b64 %rd<44>;\n"
ANCHOR_NTOK = "    ld.param.u32 %r3, [p_ntok];\n"

INSERT_REGS = """    // Ceiling Sprint Change 1B: 2D grid over (out_dim, ntok). Named, not
    // numbered, so they cannot collide with the existing %rN/%rdN allocation
    // (ptx-writing.md rules 2-3).
    .reg .b32 %r_gy_t0, %r_gy_nb, %r_gy_rem;
    .reg .b64 %rd_gy_xo, %rd_gy_so, %rd_gy_yo;
"""

INSERT_BODY = """
    // --- Ceiling Sprint Change 1B: 2D grid over (out_dim, ntok) ------------
    // This block owns token rows [ctaid.y*64, ctaid.y*64 + 64). Rebasing the
    // three token-indexed pointers and clamping ntok right here reproduces
    // exactly what runner.rs's serial host chunk loop did per chunk, so every
    // instruction below is unchanged: the MMA math, the m8n8k16 fragment
    // layout, the per-32-K scale epilogue and the accumulators all still see
    // "a block with `ntok` rows starting at x_qs".
    //
    // Launch contract becomes: grid = (ceil(out/64), ceil(ntok/64), 1).
    // With grid.y == 1 (any ntok <= 64) t0 is 0 and this is a no-op, so the
    // small shapes are neutral by construction, not by luck.
    //
    // t0 is a multiple of 64 (hence of 8), so t0 + round8(nn) <= round8(ntok):
    // the existing "x rows allocated to round8(ntok)" contract is exactly
    // sufficient and does not widen.
    mov.u32 %r_gy_t0, %ctaid.y;
    shl.b32 %r_gy_t0, %r_gy_t0, 6;          // t0 = ctaid.y * 64
    mul.wide.u32 %rd_gy_xo, %r_gy_t0, %r2;  // t0 * in_dim (int8, 1 B/elem)
    add.s64 %rd3, %rd3, %rd_gy_xo;          // x_qs     += t0 rows
    shr.u32 %r_gy_nb, %r2, 5;               // nb = in_dim / 32
    mul.wide.u32 %rd_gy_so, %r_gy_t0, %r_gy_nb;
    shl.b64 %rd_gy_so, %rd_gy_so, 2;        // * 4 B per f32 activation scale
    add.s64 %rd4, %rd4, %rd_gy_so;          // x_scales += t0 rows
    mul.wide.u32 %rd_gy_yo, %r_gy_t0, %r1;  // t0 * out_dim
    shl.b64 %rd_gy_yo, %rd_gy_yo, 2;        // * 4 B per f32 output element
    add.s64 %rd5, %rd5, %rd_gy_yo;          // y        += t0 rows
    sub.s32 %r_gy_rem, %r3, %r_gy_t0;       // rows remaining after t0
    max.s32 %r_gy_rem, %r_gy_rem, 0;        // grid.y is sized so this cannot
                                            // go negative; free insurance
    min.s32 %r3, %r_gy_rem, 64;             // ntok := this block's row count
    // --- end Change 1B -----------------------------------------------------
"""


def die(msg):
    print("patch_ptx_2d_grid.py: ABORT -- {}".format(msg), file=sys.stderr)
    print("No file was written.", file=sys.stderr)
    sys.exit(1)


def main():
    if len(sys.argv) != 2:
        die("usage: patch_ptx_2d_grid.py <path/to/glcuda_sm75.ptx>")
    path = sys.argv[1]

    with open(path, "r", encoding="ascii", newline="") as f:
        src = f.read()

    if "%ctaid.y" in src:
        die("PTX already references %ctaid.y -- already patched? Refusing to "
            "double-patch. Restore from the .bak and re-run.")

    # Scope to gl_gemm_mma_q8's body: everything before the r256 entry.
    i_target = src.find(ENTRY_TARGET)
    i_r256 = src.find(ENTRY_R256)
    if i_target < 0:
        die("could not find '{}' -- has the kernel been renamed?".format(ENTRY_TARGET))
    if i_r256 < 0:
        die("could not find '{}' -- file structure changed; the anchor-scoping "
            "assumption no longer holds, so a naive patch could corrupt the "
            "second kernel. Re-read the PTX before proceeding.".format(ENTRY_R256))
    if i_r256 < i_target:
        die("gl_gemm_mma_q8_r256 appears BEFORE gl_gemm_mma_q8; scoping "
            "assumption inverted. Re-read the PTX.")

    head, tail = src[:i_r256], src[i_r256:]

    for name, anchor in (("register declaration", ANCHOR_REGS),
                         ("ld.param ntok", ANCHOR_NTOK)):
        n = head.count(anchor)
        if n != 1:
            die("expected exactly 1 '{}' anchor in gl_gemm_mma_q8's body, "
                "found {}. Anchor text or formatting changed -- fix this "
                "script, do not patch by hand.".format(name, n))

    # Registers first, then the body. Order matters only for readability.
    head = head.replace(ANCHOR_REGS, ANCHOR_REGS + INSERT_REGS, 1)
    head = head.replace(ANCHOR_NTOK, ANCHOR_NTOK + INSERT_BODY, 1)
    out = head + tail

    # ptx-writing.md rule 1: pure ASCII, LF endings. A single smart quote or
    # em-dash makes ptxas reject the whole module before parsing one
    # instruction -- this has actually happened in this project.
    try:
        out.encode("ascii")
    except UnicodeEncodeError as e:
        die("patched PTX is not pure ASCII: {}".format(e))
    if "\r" in out:
        die("patched PTX contains CR -- must be LF-only")

    with open(path, "w", encoding="ascii", newline="\n") as f:
        f.write(out)

    added = out.count("\n") - src.count("\n")
    print("patch_ptx_2d_grid.py: OK")
    print("  patched : gl_gemm_mma_q8   (2D grid, ctaid.y token-row offset)")
    print("  untouched: gl_gemm_mma_q8_r256 (known numerical bug -- reported, "
          "not optimized)")
    print("  +{} lines, pure ASCII, LF endings".format(added))
    print("  new launch contract: grid = (ceil(out/64), ceil(ntok/64), 1)")


if __name__ == "__main__":
    main()


## 5. Build the cuda-oxide kernel crate

Extracts the PTX cuda-oxide generates. Per the architecture docs, output lands next to the host binary: `target/<profile>/<bin-name>.ptx`.

**Confirmed on real hardware**: `cargo oxide build`'s usage is `cargo oxide build [OPTIONS] [EXAMPLE] [-- [CARGO_ARGS]...]` -- `--release` is a cargo flag, not a `cargo-oxide` option, so it must go after `--` (`cargo oxide build -- --release`), not bare (`cargo oxide build --release` errors with "unexpected argument"). `EXAMPLE` is optional when run from inside the crate's own directory.

In [ ]:
%%bash
set -e
source "$HOME/.cargo/env"
cd gwenland-kernel-research/kernels
cargo oxide build --help
echo '--- usage above confirms: EXAMPLE is a positional (optional when'
echo 'run from inside the crate dir), cargo flags go after -- ---'
cargo oxide build -- --release
echo '--- everything named *.ptx anywhere under this crate (no'
echo '-newer filter -- that filter is what actually failed the first'
echo 'time this notebook ran to this point: cargo-oxide places its'
echo 'output somewhere ptxas/cargo mtime bookkeeping does not treat'
echo "as 'newer than Cargo.toml', so the filtered search came up"
echo 'empty even though the build had genuinely succeeded) ---'
find . -name '*.ptx'


In [ ]:
%%bash
cd gwenland-kernel-research/kernels
PTX=$(find . -name '*.ptx' | head -1)
if [ -z "$PTX" ]; then
  echo 'No .ptx found anywhere under kernels/. Build likely failed'
  echo 'silently or cargo-oxide emits under a name/location Cell 5'
  echo "did not anticipate -- check Cell 5's full output above,"
  echo 'and try: find / -name gwenland-kernels.ptx 2>/dev/null'
  exit 1
fi
echo "Found: $PTX"
cp "$PTX" gwenland-kernels.ptx
wc -l gwenland-kernels.ptx
grep -n '.visible .entry' gwenland-kernels.ptx


**Fixed after the first real compile on a T4** (2026-08-04) -- `kernels/src/lib.rs` originally called `DisjointSlice::get_mut(usize)`, which doesn't exist (`get_mut` wants a `ThreadIndex`, only mintable from `thread::index_1d()`/`index_2d()` -- this kernel's per-thread writes don't fit that one-index-per-thread model). Switched to `DisjointSlice::get_unchecked_mut` (the crate's documented escape hatch for "scatter operations with known-unique destinations", `crates/cuda-device/src/disjoint.rs:292`). Chasing that down the real PTX comments also turned up two silent correctness bugs the compiler never would have caught: a shared-instead-of-per-column dequant scale, and a missing predicate gate that let inactive warps race each other for column 0. All three are fixed in the source below and commented in place. Everything else in the file (`threadIdx_x`, `blockDim_x`, `blockIdx_x`, `sync_threads`, `SharedArray`, `cuda_device::wmma::mma_m8n8k16_s32_s8`) compiled clean on the first try -- only `get_mut` errored.

Re-run Cell 4 onward (the workspace now writes the fixed source) before Cell 5's build.

**Real finding from reading the generated PTX** (2026-08-04, first successful build): `gl_gemm_mma_q8_oxide` compiles to `.local .align 4 .b8 __local_depot0[264]` -- the `acc: [[f32;2];32]` accumulator is **not in registers**. Because `m` is a *runtime* loop variable, LLVM cannot promote a dynamically-indexed 32-entry array to SSA/register form, so it spills the whole thing (plus the `wsc0`/`wsc1` scale temporaries) to `.local` memory -- `ld.local`/`st.local` on every single MMA iteration (`nb * m_tiles` times per launch). The hand kernel never pays this: its accumulators are named PTX registers (`.reg .f32 %f<32>` / `%f<80>`) end to end. This is real evidence for the module doc's "known gap" note -- a `const M_TILES: u32` version (compile-time loop bound, so `m` is either fully unrolled or at least structurally analyzable) is the natural next experiment, not just a hedge anymore. Expect this to matter more for throughput than the register-count comparison alone would suggest -- `.local` traffic is extra memory ops in the hot loop, not just register pressure.

Also worth keeping for the report: `setp.gt.u32 %p28, %r112, 31; @%p28 bra ...trap;` is Rust's automatic bounds check on `acc[m as usize]` compiling to a real trap if `m_tiles` is ever launched above `MAX_M_TILES`. The hand kernel has no equivalent -- that misuse would silently corrupt memory there instead. A genuine, unprompted safety win from the port, worth noting in Task 5's findings regardless of which kernel wins on speed.

## 6. Fetch the reference hand-written PTX

This URL was verified during research to serve the exact file this notebook is being benchmarked against -- confirmed via `git diff github/engine/gljax-bringup -- glcuda/src/kernels/glcuda_sm75.ptx` (empty diff) before use, not guessed.

In [ ]:
%%bash
set -e
mkdir -p gwenland-kernel-research/runner/reference
curl -sSf https://raw.githubusercontent.com/gwenland-org/gwenland-ai/engine/gljax-bringup/glcuda/src/kernels/glcuda_sm75.ptx -o gwenland-kernel-research/runner/reference/glcuda_sm75.ptx
wc -l gwenland-kernel-research/runner/reference/glcuda_sm75.ptx
echo
echo 'If this branch has moved since 2026-08-04, this file may differ'
echo 'from what Task 1 audited -- re-diff against your local checkout'
echo 'if the kernel count/names below look different from:'
echo '  gl_gemm_mma_q8, gl_gemm_mma_q8_r256'
grep -n '.visible .entry' gwenland-kernel-research/runner/reference/glcuda_sm75.ptx


## 6b. Ceiling Sprint Change 1B -- patch the PTX for a 2D grid

Phase 2's approved change: `grid.y = ceil(ntok/64)`, kernel derives its token-row offset from `%ctaid.y`. This moves the 64-row chunk loop that `glcuda/src/runner.rs:166-180` runs on the HOST into the grid, so case 3 goes from **4 sequential launches over 8 blocks (8/40 SMs)** to **one launch over 32 blocks (32/40 SMs)** -- and the 4 token-chunks now share one 2.2 MB weight matrix concurrently, which fits the T4's 4 MB L2, so the redundant weight reads Phase 1 measured (intensity 273 -> 101 ops/byte) become L2 hits instead of DRAM re-reads.

**Research copy only.** `.bak` is kept as the A/B baseline and the monorepo PTX is untouched until Gate 3 confirms a win.

The patch script was written against the real PTX and dry-run locally before this notebook shipped: both of its anchors occur **twice** in the file (once in `gl_gemm_mma_q8`, once in `gl_gemm_mma_q8_r256`), so it slices at the r256 entry and asserts exactly one match in the target body. `gl_gemm_mma_q8_r256` is deliberately **not** patched -- Phase 1 finding 4 showed it computes wrong numbers on aligned synthetic buffers, and optimizing a broken kernel produces meaningless timings.

In [ ]:
%%bash
set -e
PTX=gwenland-kernel-research/runner/reference/glcuda_sm75.ptx
# .bak is the A/B baseline the runner loads as the 1D-chunked path --
# not just a safety copy. Only make it on the first run, so re-running
# this cell can never overwrite the pristine baseline with a patched one.
if [ ! -f "$PTX.bak" ]; then cp "$PTX" "$PTX.bak"; fi
cp "$PTX.bak" "$PTX"
python3 gwenland-kernel-research/patch_ptx_2d_grid.py "$PTX"
echo
echo '--- ctaid.y must appear ONLY in gl_gemm_mma_q8 (before the r256 entry) ---'
grep -n 'ctaid.y\|.visible .entry' "$PTX"


## 7. Register / shared-memory usage (`ptxas -v`)

Real numbers, not the hand kernels' self-estimates in their header comments -- `gl_gemm_mma_q8_r256`'s comment claims "~92 registers"; `glcuda/src/runner.rs:156` cites a measured "96 reg / 0 spill" from an earlier `ptxas -v` run. Reproduce both here alongside the cuda-oxide kernel's numbers for a same-methodology comparison.

In [ ]:
%%bash
set -e
REF=gwenland-kernel-research/runner/reference
# Assemble BOTH copies. This is the first place a malformed patch would
# surface: ptxas rejects a bad module outright, and it must happen HERE
# rather than as a cryptic module-load error inside the benchmark.
echo '=== BASELINE (.bak, 1D grid) ==='
ptxas -arch=sm_75 -v "$REF/glcuda_sm75.ptx.bak" -o /tmp/hand_base.cubin 2>&1 | grep -A3 'gl_gemm_mma_q8'
echo
echo '=== PATCHED (2D grid, Change 1B) ==='
ptxas -arch=sm_75 -v "$REF/glcuda_sm75.ptx" -o /tmp/hand_2d.cubin 2>&1 | grep -A3 'gl_gemm_mma_q8'
echo
echo 'Compare gl_gemm_mma_q8 registers/smem above: the patch adds 6 scalar'
echo 'registers of index math and no shared memory, so a LARGE register'
echo 'jump (or any spill appearing) is a red flag worth reading before'
echo 'trusting the timings. r256 must be byte-identical in both -- it was'
echo 'deliberately not patched.'
echo
echo '=== cuda-oxide: gl_gemm_mma_q8_oxide ==='
ptxas -arch=sm_75 -v gwenland-kernel-research/kernels/gwenland-kernels.ptx -o /tmp/oxide.cubin 2>&1 | grep -A3 'gl_gemm_mma_q8_oxide'


## 8. Occupancy from the measured register/smem numbers

Plain arithmetic against the T4 (sm_75) ceilings from the task brief -- 65536 registers/SM, 64KB shared/SM (96KB with carveout), 1024 threads/SM, 32 blocks/SM max. Fill in REGS_PER_THREAD and SMEM_PER_BLOCK per kernel from Cell 7's output above.

In [ ]:
def occupancy(regs_per_thread, smem_per_block, threads_per_block=256):
    REG_FILE = 65536
    SMEM_PER_SM = 64 * 1024
    MAX_THREADS_SM = 1024
    MAX_BLOCKS_SM = 32
    by_regs = REG_FILE // (regs_per_thread * threads_per_block) if regs_per_thread else MAX_BLOCKS_SM
    by_smem = SMEM_PER_SM // smem_per_block if smem_per_block else MAX_BLOCKS_SM
    by_threads = MAX_THREADS_SM // threads_per_block
    blocks_per_sm = min(by_regs, by_smem, by_threads, MAX_BLOCKS_SM)
    occ = blocks_per_sm * threads_per_block / MAX_THREADS_SM
    print(f'regs/thread={regs_per_thread} smem/block={smem_per_block}B -> '
          f'{blocks_per_sm} blocks/SM, {occ*100:.0f}% occupancy '
          f'(limited by: {"regs" if by_regs==blocks_per_sm else "smem" if by_smem==blocks_per_sm else "threads"})')
    return occ

# Real ptxas -v numbers, measured on a T4, 2026-08-04 (Cell 7's actual
# output -- these are no longer placeholders):
print('gl_gemm_mma_q8        (hand, PRODUCTION, 8-tile):')
occupancy(regs_per_thread=44, smem_per_block=2304)
print('gl_gemm_mma_q8_r256   (hand, 32-tile, NOT in production):')
occupancy(regs_per_thread=96, smem_per_block=9216)
print('gl_gemm_mma_q8_oxide  (cuda-oxide -- ONE compiled kernel body')
print('                       serves BOTH m_tiles=8 and m_tiles=32')
print('                       launches, so this is the same number')
print('                       for both comparisons below):')
occupancy(regs_per_thread=50, smem_per_block=9216)
print()
print('STRIKING RESULT: cuda-oxide matches the hand 8-tile kernel')
print('100% occupancy despite 4x its shared-memory footprint and a')
print('264 B/thread stack frame it does not have, AND *doubles* the')
print('hand 32-tile kernel occupancy (100% vs 50%) -- because its')
print('register count (50) is barely above the 8-tile kernel (44)')
print('and well under the 32-tile kernel (96). Very likely the SAME')
print('mechanism as the .local accumulator spill flagged after Cell')
print('5/6: pushing the accumulator out of registers is probably')
print('*why* register pressure stays nearly flat as m_tiles goes')
print('8->32, which is exactly what buys the occupancy headroom the')
print('hand 32-tile kernel does not have. Occupancy is not')
print('throughput though (kernel-design.md rule 3) -- the .local')
print('traffic this trades for is real extra memory ops per MMA')
print('iteration. Cell 9 settles it with actual tok/s, not this.')


## 9. Build + run the benchmark/correctness harness

`runner/` is plain stable Rust (cudarc only) -- does not need the nightly toolchain from Cell 2. Loads both PTX files uniformly and prints the Task 3/4 report: throughput, TOPS-vs-ceiling, and correctness against the scalar CPU reference for all four kernel launches, across three shapes including the 896 (non-power-of-2, real Qwen2.5-0.5B) case.

**Learned the hard way**: `%%writefile` only takes effect when *that specific cell* re-runs -- re-running just the build cell after fixing `runner/src/main.rs` above silently rebuilds the STALE file still on disk (this happened once already in this research pass: an identical error, same line numbers, on a file that had already been fixed several cells up). The next cell re-writes `runner/`'s two files immediately before the build, every time, so this can't recur -- always run it, don't skip straight to the build cell below it.

In [ ]:
%%writefile gwenland-kernel-research/runner/Cargo.toml
[package]
name = "gwenland-kernel-runner"
version = "0.1.0"
edition = "2021"

# runner/ is plain stable-toolchain-compatible Rust -- it only LOADS and
# LAUNCHES already-compiled .ptx files via cudarc's driver API. It never
# compiles device code itself, so it does not need cuda-oxide's nightly
# toolchain. This is deliberate: cuda-oxide is a PTX *generator* here, and
# the benchmark harness stays uniform across "hand-written PTX" and
# "cuda-oxide-generated PTX" -- both are just PTX files to cudarc.
[dependencies]
cudarc = { version = "0.19.8", default-features = false, features = [
    "driver",
    "cuda-version-from-build-system",
    # default-features=false drops "fallback-dynamic-loading" too (it's a
    # DEFAULT feature, not something on regardless) -- cudarc's build.rs
    # panics with "None between dynamic-loading, fallback-dynamic-loading,
    # dynamic-linking and static-linking features are active, this is a bug"
    # without one of these explicitly re-added. fallback-dynamic-loading
    # (dlopen libcuda.so at runtime, falls back gracefully if absent) is the
    # right choice for a Kaggle T4 instance where we don't control how CUDA
    # was installed.
    "fallback-dynamic-loading",
    # `Ptx` (from_file/from_src) AND `CudaContext::load_module` are BOTH
    # `#[cfg(feature = "nvrtc")]`-gated in cudarc's source (driver/safe/
    # core.rs:2173) even though we never call nvrtc's actual runtime
    # compiler (compile_ptx) -- the feature gates the whole `nvrtc` module,
    # Ptx type included, not just the compilation entry points. Confirmed
    # by reading cudarc v0.19.8 source directly, not from docs.
    "nvrtc",
] }
rand = "0.8"


In [ ]:
%%writefile gwenland-kernel-research/runner/src/main.rs
//! Ceiling Sprint Phase 3 harness — Changes 1A + 1B.
//!
//! CHANGE 1A (re-baseline). Every variant is now measured as
//! `1 warmup launch -> correctness check -> ITERS launches -> ONE sync`,
//! matching `glcuda/examples/bench.rs`'s existing `[mma gate]` pattern.
//! The previous version timed a single COLD launch with a `synchronize()`
//! inside the timed region, which measured launch overhead, not the kernel:
//! Phase 1 showed case 1 and case 2 differ by 224x in work but only 1.9x in
//! measured time, which is impossible for a kernel-bound measurement.
//!
//! CHANGE 1B (2D grid). The hand kernel is now measured on BOTH paths:
//!   * `[1D chunked]`  original PTX + the host-side 64-row chunk loop that
//!                     `glcuda/src/runner.rs:166-180` actually runs in
//!                     production. This is the honest baseline -- it
//!                     includes the chunking cost Phase 1 quantified
//!                     (intensity 273 -> 101 ops/byte).
//!   * `[2D grid]`     patched PTX, ONE launch, grid.y = ceil(ntok/64).
//! Loading both the original and the patched module (rather than reusing the
//! patched one with grid.y=1 for the baseline) means the A/B also catches any
//! cost the patch itself introduces.
//!
//! `gl_gemm_mma_q8_r256` is measured for CORRECTNESS ONLY -- Phase 1 finding
//! 4 established it computes wrong numbers on aligned synthetic buffers, and
//! timing a broken kernel is meaningless.
//!
//! Reminder for reading the output: a win here is DIAGNOSTIC ONLY. Per Phase
//! 2, the merge gate is glbench `prefill_tps` on a real model. This repo has
//! twice recorded a ~2x isolated win that was neutral in production
//! (VNNI-512, row-tile GEMM).

use cudarc::driver::{
    CudaContext, CudaFunction, CudaSlice, CudaStream, DevicePtr, DevicePtrMut, LaunchConfig,
    PushKernelArg,
};
use cudarc::nvrtc::Ptx;
use std::path::PathBuf;
// cudarc's allocation and host<->device copy methods (`alloc_zeros`,
// `clone_htod`, `clone_dtoh`) take `self: &Arc<Self>`, NOT `&self`
// (driver/safe/core.rs:1559-1631), so every helper that copies has to hold
// the Arc rather than a plain &CudaStream. `launch_builder`, `synchronize`
// and `device_ptr` are ordinary `&self` methods and deref-coerce from the
// Arc for free.
use std::sync::Arc;
use std::time::{Duration, Instant};

/// Tolerance for the INT8 tensor-core GEMM vs the scalar dequant reference.
/// Wide because the s32 accumulate is exact but the f16 weight-scale round
/// trip is not; the sprint brief fixes this figure at 5.0e-2.
const TOL_MATMUL: f32 = 5.0e-2;

/// Measured iterations per variant (matches bench.rs's `[mma gate]`).
const ITERS: u32 = 50;

/// T4 spec figures, for reporting only.
const T4_SMS: u32 = 40;
const T4_INT8_TOPS: f64 = 65.0;

struct Q8Case {
    out_dim: usize,
    in_dim: usize,
    ntok: usize,
}

fn quantize_q8_0(rows: usize, cols: usize, data: &[f32]) -> (Vec<i8>, Vec<f32>, usize) {
    let nb = cols / 32;
    let mut qs = vec![0i8; rows * cols];
    let mut scales = vec![0f32; rows * nb];
    for r in 0..rows {
        for b in 0..nb {
            let base = r * cols + b * 32;
            let block = &data[base..base + 32];
            let amax = block.iter().fold(0f32, |m, &v| m.max(v.abs()));
            let scale = if amax > 0.0 { amax / 127.0 } else { 1.0 };
            scales[r * nb + b] = scale;
            for i in 0..32 {
                qs[base + i] = (block[i] / scale).round().clamp(-127.0, 127.0) as i8;
            }
        }
    }
    (qs, scales, nb)
}

fn scalar_reference(
    w_qs: &[i8],
    w_scales: &[f32],
    x_qs: &[i8],
    x_scales: &[f32],
    out_dim: usize,
    in_dim: usize,
    ntok: usize,
) -> Vec<f32> {
    let nb = in_dim / 32;
    let mut y = vec![0f32; ntok * out_dim];
    for t in 0..ntok {
        for o in 0..out_dim {
            let mut acc = 0f32;
            for b in 0..nb {
                let wsc = w_scales[o * nb + b];
                let xsc = x_scales[t * nb + b];
                let mut dot = 0i32;
                for k in 0..32 {
                    dot += w_qs[o * in_dim + b * 32 + k] as i32
                        * x_qs[t * in_dim + b * 32 + k] as i32;
                }
                acc += dot as f32 * wsc * xsc;
            }
            y[t * out_dim + o] = acc;
        }
    }
    y
}

fn f32_to_f16_bits(v: f32) -> u16 {
    let bits = v.to_bits();
    let sign = (bits >> 16) & 0x8000;
    let exp = ((bits >> 23) & 0xFF) as i32 - 127 + 15;
    let frac = bits & 0x7FFFFF;
    let half = if exp <= 0 {
        0
    } else if exp >= 0x1F {
        0x7C00
    } else {
        ((exp as u32) << 10) | (frac >> 13)
    };
    (sign | half) as u16
}

fn max_abs_diff(a: &[f32], b: &[f32]) -> f32 {
    a.iter()
        .zip(b.iter())
        .map(|(x, y)| (x - y).abs())
        .fold(0f32, f32::max)
}

// ---------------------------------------------------------------------------
// Launch paths
// ---------------------------------------------------------------------------

/// Production-equivalent baseline: 1-D grid, host-side 64-row chunk loop.
/// Mirrors `glcuda/src/runner.rs:166-180` exactly, including the fact that
/// each chunk re-reads the entire weight matrix.
#[allow(clippy::too_many_arguments)]
fn launch_hand_1d_chunked(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    gx: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) {
    let nb = in_dim / 32;
    let cfg = LaunchConfig {
        grid_dim: (gx, 1, 1),
        block_dim: (256, 1, 1),
        shared_mem_bytes: 0,
    };
    let mut t0 = 0u32;
    while t0 < ntok {
        let nn = (ntok - t0).min(64);
        let xq = d_xqs.slice((t0 * in_dim) as usize..);
        let xs = d_xsc.slice((t0 * nb) as usize..);
        let mut yv = d_y.slice_mut((t0 * out_dim) as usize..);
        let mut b = stream.launch_builder(f);
        b.arg(d_wqs);
        b.arg(d_wsc);
        b.arg(&xq);
        b.arg(&xs);
        b.arg(&mut yv);
        b.arg(&out_dim);
        b.arg(&in_dim);
        b.arg(&nn);
        // Binding silences Option's #[must_use]; it is None unless
    // record_kernel_launch() requested built-in event timing (we time on the
    // host, around the whole 50-iteration loop).
    let _evt = unsafe { b.launch(cfg) }.unwrap();
        t0 += nn;
    }
}

/// Single launch with an EXPLICIT grid.
///
/// `gy` is a parameter, not derived, because two different callers need
/// different values and getting it wrong is silent:
///   * Change 1B candidate (patched kernel): `gy = ceil(ntok/64)`.
///   * `gl_gemm_mma_q8_r256` (NOT patched): `gy = 1` is mandatory. It has no
///     `%ctaid.y` read, so every block would ignore its y index and
///     recompute the same rows into the same output -- duplicate work and
///     racing writes, not a crash. Deriving `gy` inside this function once
///     did exactly that to r256.
#[allow(clippy::too_many_arguments)]
fn launch_hand_single(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    gx: u32,
    gy: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) {
    let cfg = LaunchConfig {
        grid_dim: (gx, gy.max(1), 1),
        block_dim: (256, 1, 1),
        shared_mem_bytes: 0,
    };
    let mut b = stream.launch_builder(f);
    b.arg(d_wqs);
    b.arg(d_wsc);
    b.arg(d_xqs);
    b.arg(d_xsc);
    b.arg(d_y);
    b.arg(&out_dim);
    b.arg(&in_dim);
    b.arg(&ntok);
    // Binding silences Option's #[must_use]; it is None unless
    // record_kernel_launch() requested built-in event timing (we time on the
    // host, around the whole 50-iteration loop).
    let _evt = unsafe { b.launch(cfg) }.unwrap();
}

/// cuda-oxide kernel. Needs an explicit (ptr, len) pair per slice: cudarc's
/// `.arg(&CudaSlice)` pushes only the pointer, but cuda-oxide compiles Rust
/// `&[T]` to a fat pointer, so pointer-only args desync every later
/// parameter (this was a real segfault earlier in this research pass).
#[allow(clippy::too_many_arguments)]
fn launch_oxide(
    stream: &Arc<CudaStream>,
    f: &CudaFunction,
    cfg: LaunchConfig,
    m_tiles: u32,
    d_wqs: &CudaSlice<i8>,
    d_wsc: &CudaSlice<u16>,
    d_xqs: &CudaSlice<i8>,
    d_xsc: &CudaSlice<f32>,
    d_y: &mut CudaSlice<f32>,
    out_dim: u32,
    in_dim: u32,
    ntok: u32,
) -> Result<(), cudarc::driver::DriverError> {
    let wqs_len = d_wqs.len() as u64;
    let wsc_len = d_wsc.len() as u64;
    let xqs_len = d_xqs.len() as u64;
    let xsc_len = d_xsc.len() as u64;
    let y_len = d_y.len() as u64;
    let (wqs_ptr, _g1) = d_wqs.device_ptr(stream);
    let (wsc_ptr, _g2) = d_wsc.device_ptr(stream);
    let (xqs_ptr, _g3) = d_xqs.device_ptr(stream);
    let (xsc_ptr, _g4) = d_xsc.device_ptr(stream);
    let (y_ptr, _g5) = d_y.device_ptr_mut(stream);
    let mut b = stream.launch_builder(f);
    b.arg(&out_dim);
    b.arg(&in_dim);
    b.arg(&ntok);
    b.arg(&m_tiles);
    b.arg(&wqs_ptr);
    b.arg(&wqs_len);
    b.arg(&wsc_ptr);
    b.arg(&wsc_len);
    b.arg(&xqs_ptr);
    b.arg(&xqs_len);
    b.arg(&xsc_ptr);
    b.arg(&xsc_len);
    b.arg(&y_ptr);
    b.arg(&y_len);
    // `launch` returns Result<Option<(CudaEvent, CudaEvent)>, _> -- the
    // Option is Some only when `record_kernel_launch()` asked for built-in
    // event timing, which this harness does not (it times on the host
    // around the whole 50-iteration loop). Discard the None.
    unsafe { b.launch(cfg) }.map(|_| ())
}

// ---------------------------------------------------------------------------
// Measurement (Change 1A)
// ---------------------------------------------------------------------------

/// warmup -> correctness -> ITERS timed launches -> one sync.
/// The correctness check runs ONCE, on the warmup result, so the timed loop
/// measures only kernel time (no device-to-host copies inside it).
fn measure(
    stream: &Arc<CudaStream>,
    d_y: &mut CudaSlice<f32>,
    reference: &[f32],
    launch: &mut dyn FnMut(&mut CudaSlice<f32>),
) -> (Duration, f32) {
    launch(d_y);
    stream.synchronize().unwrap();
    let y_host = stream.clone_dtoh(d_y).unwrap();
    let diff = max_abs_diff(&y_host, reference);

    let t0 = Instant::now();
    for _ in 0..ITERS {
        launch(d_y);
    }
    stream.synchronize().unwrap();
    (t0.elapsed() / ITERS, diff)
}

#[allow(clippy::too_many_arguments)]
fn report(label: &str, dt: Duration, diff: f32, case: &Q8Case, blocks: u32, baseline: Option<Duration>) {
    let ops = 2.0 * case.out_dim as f64 * case.in_dim as f64 * case.ntok as f64;
    let tops = ops / dt.as_secs_f64() / 1e12;
    let status = if diff < TOL_MATMUL { "PASS" } else { "FAIL" };
    let sms = blocks.min(T4_SMS);
    let delta = match baseline {
        Some(b) if b.as_secs_f64() > 0.0 => format!(
            "  delta {:+.1}%",
            (b.as_secs_f64() / dt.as_secs_f64() - 1.0) * 100.0
        ),
        _ => String::new(),
    };
    println!(
        "  {label:<34} {:>8.2}us  {:>6.3} TOPS ({:>4.1}% of 65)  blocks={blocks:<3} ({sms}/{T4_SMS} SM)  max_abs_diff={diff:.3e} [{status}]{delta}",
        dt.as_secs_f64() * 1e6,
        tops,
        tops / T4_INT8_TOPS * 100.0,
    );
}

fn main() {
    let hand_ptx = std::env::var("HAND_PTX")
        .unwrap_or_else(|_| "reference/glcuda_sm75.ptx".to_string());
    let hand_ptx_orig = std::env::var("HAND_PTX_ORIG")
        .unwrap_or_else(|_| "reference/glcuda_sm75.ptx.bak".to_string());
    let oxide_ptx = std::env::var("OXIDE_PTX")
        .unwrap_or_else(|_| "../kernels/gwenland-kernels.ptx".to_string());

    let ctx = CudaContext::new(0).unwrap();
    let stream = ctx.default_stream();

    let m_orig = ctx
        .load_module(Ptx::from_file(PathBuf::from(&hand_ptx_orig)))
        .unwrap();
    let m_2d = ctx
        .load_module(Ptx::from_file(PathBuf::from(&hand_ptx)))
        .unwrap();
    let m_oxide = ctx
        .load_module(Ptx::from_file(PathBuf::from(&oxide_ptx)))
        .unwrap();

    let f_1d = m_orig.load_function("gl_gemm_mma_q8").unwrap();
    let f_r256 = m_orig.load_function("gl_gemm_mma_q8_r256").unwrap();
    let f_2d = m_2d.load_function("gl_gemm_mma_q8").unwrap();
    let f_oxide = m_oxide.load_function("gl_gemm_mma_q8_oxide").unwrap();

    println!("=== CEILING SPRINT -- GATE 3 RESULTS ===");
    println!("Hardware: T4, Kaggle");
    println!(
        "Methodology: 1 warmup launch + {ITERS} measured iterations, single sync after loop"
    );
    println!("Baseline path: original PTX + host 64-row chunk loop (== production runner.rs)");
    println!("Candidate path: patched PTX, single launch, grid.y = ceil(ntok/64)");

    // dim 896 (Qwen2.5-0.5B) is deliberately NOT a power of two; ntok=200 is
    // deliberately NOT a multiple of 64, to exercise the grid.y tail block.
    let cases = [
        Q8Case { out_dim: 64, in_dim: 128, ntok: 8 },
        Q8Case { out_dim: 256, in_dim: 896, ntok: 64 },
        Q8Case { out_dim: 512, in_dim: 4096, ntok: 256 },
        Q8Case { out_dim: 256, in_dim: 896, ntok: 200 },
    ];

    for case in &cases {
        println!(
            "\n=== case out={} in={} ntok={} ===",
            case.out_dim, case.in_dim, case.ntok
        );
        let ntok_pad8 = (case.ntok + 7) & !7;
        let out_u32 = case.out_dim as u32;
        let in_u32 = case.in_dim as u32;
        let ntok_u32 = case.ntok as u32;
        let gx = out_u32.div_ceil(64).max(1);
        let gy = ntok_u32.div_ceil(64).max(1);

        let mut rng_state = 0x2026_08_04u64;
        let mut next = || {
            rng_state ^= rng_state << 13;
            rng_state ^= rng_state >> 7;
            rng_state ^= rng_state << 17;
            ((rng_state % 2000) as f32 / 1000.0) - 1.0
        };
        let w_f32: Vec<f32> = (0..case.out_dim * case.in_dim).map(|_| next()).collect();
        let x_f32: Vec<f32> = (0..ntok_pad8 * case.in_dim).map(|_| next()).collect();

        let (w_qs, w_scales_f32, _) = quantize_q8_0(case.out_dim, case.in_dim, &w_f32);
        let (x_qs, x_scales, _) = quantize_q8_0(ntok_pad8, case.in_dim, &x_f32);
        let w_scales_f16: Vec<u16> = w_scales_f32.iter().map(|&s| f32_to_f16_bits(s)).collect();
        let reference = scalar_reference(
            &w_qs, &w_scales_f32, &x_qs, &x_scales, case.out_dim, case.in_dim, case.ntok,
        );

        let d_wqs = stream.clone_htod(&w_qs).unwrap();
        let d_wsc16 = stream.clone_htod(&w_scales_f16).unwrap();
        let d_xqs = stream.clone_htod(&x_qs).unwrap();
        let d_xsc = stream.clone_htod(&x_scales).unwrap();
        let mut d_y = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();

        // --- baseline: 1D grid + host chunk loop (production-equivalent) ---
        let (dt_1d, diff_1d) = {
            let mut f = |y: &mut CudaSlice<f32>| {
                launch_hand_1d_chunked(
                    &stream, &f_1d, gx, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y, out_u32, in_u32,
                    ntok_u32,
                )
            };
            measure(&stream, &mut d_y, &reference, &mut f)
        };
        report("[1D chunked] gl_gemm_mma_q8", dt_1d, diff_1d, case, gx, None);

        // --- candidate: 2D grid, single launch ---
        let (dt_2d, diff_2d) = {
            let mut f = |y: &mut CudaSlice<f32>| {
                launch_hand_single(
                    &stream, &f_2d, gx, gy, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y, out_u32, in_u32,
                    ntok_u32,
                )
            };
            measure(&stream, &mut d_y, &reference, &mut f)
        };
        report("[2D grid]    gl_gemm_mma_q8", dt_2d, diff_2d, case, gx * gy, Some(dt_1d));

        if gy == 1 {
            println!("      (grid.y=1 -> ctaid.y offset is a no-op; neutral is the EXPECTED result)");
        }
        if (diff_1d - diff_2d).abs() > 0.0 {
            println!(
                "      NOTE: 1D and 2D max_abs_diff differ ({diff_1d:.6e} vs {diff_2d:.6e}) -- \
                 for grid.y=1 cases they should be byte-identical; investigate before trusting timings"
            );
        }

        // --- r256: correctness only (Phase 1 finding 4) ---
        if case.ntok <= 256 {
            let mut yv = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();
            // gy = 1 is REQUIRED: r256 was deliberately not patched, so it has
            // no %ctaid.y read and covers all its rows (up to 256) from a 1-D
            // grid on its own.
            launch_hand_single(
                &stream, &f_r256, gx, 1, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, &mut yv, out_u32,
                in_u32, ntok_u32,
            );
            stream.synchronize().unwrap();
            let y_host = stream.clone_dtoh(&yv).unwrap();
            let d = max_abs_diff(&y_host, &reference);
            let st = if d < TOL_MATMUL { "PASS" } else { "FAIL" };
            println!(
                "  {:<34} correctness only, not timed          max_abs_diff={d:.3e} [{st}]",
                "[1D]         gl_gemm_mma_q8_r256"
            );
        }

        // --- cuda-oxide, 1-D grid (unchanged; diagnostic context only) ---
        for &m_tiles in &[8u32, 32u32] {
            if case.ntok > (m_tiles * 8) as usize {
                continue;
            }
            let cfg = LaunchConfig {
                grid_dim: (gx, 1, 1),
                block_dim: (256, 1, 1),
                shared_mem_bytes: 0,
            };
            let mut yv = stream.alloc_zeros::<f32>(case.ntok * case.out_dim).unwrap();
            let probe = launch_oxide(
                &stream, &f_oxide, cfg, m_tiles, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, &mut yv,
                out_u32, in_u32, ntok_u32,
            );
            if let Err(e) = probe {
                println!("  [oxide m{m_tiles}] LAUNCH FAILED: {e:?}");
                continue;
            }
            let (dt, diff) = {
                let mut f = |y: &mut CudaSlice<f32>| {
                    launch_oxide(
                        &stream, &f_oxide, cfg, m_tiles, &d_wqs, &d_wsc16, &d_xqs, &d_xsc, y,
                        out_u32, in_u32, ntok_u32,
                    )
                    .unwrap()
                };
                measure(&stream, &mut yv, &reference, &mut f)
            };
            report(
                &format!("[1D oxide]   m_tiles={m_tiles}"),
                dt,
                diff,
                case,
                gx,
                Some(dt_1d),
            );
        }
    }

    println!("\n--- REMINDER ---");
    println!("These are DIAGNOSTIC numbers. Per Phase 2 the merge gate is glbench");
    println!("prefill_tps on a real model; an isolated win here does not justify");
    println!("touching the monorepo PTX (VNNI-512 and row-tile both won isolated");
    println!("and went neutral in production).");
}


In [ ]:
%%bash
set -e
source "$HOME/.cargo/env"
cd gwenland-kernel-research/runner
HAND_PTX=reference/glcuda_sm75.ptx \
HAND_PTX_ORIG=reference/glcuda_sm75.ptx.bak \
OXIDE_PTX=../kernels/gwenland-kernels.ptx \
cargo run --release


## 10. Fill in the findings report

Paste Cell 9's PASS/FAIL + TOPS numbers and Cell 7/8's register/occupancy numbers into the Task 5 report format. What's already confirmed from the first full run (2026-08-04, real T4 output) -- re-run Cell 9 after the staging-bug fix above to get corrected m_tiles=32 numbers, but the rest of this stands:

- **`gl_gemm_mma_q8_oxide` is algorithmically correct at m_tiles=8** -- `max_abs_diff` matched the hand `gl_gemm_mma_q8` kernel EXACTLY (not just within tolerance) on both cases that ran: 6.4583e-3 and 1.9100e-2. Strong evidence the Rust port is a faithful translation.
- **cuda-oxide is meaningfully SLOWER at the apples-to-apples comparison**: m_tiles=8 vs gl_gemm_mma_q8 was ~23% slower on the tiny case (39.4 vs 32.0 us) and ~3.7x slower on the 896-dim case (232.1 vs 62.7 us) -- both at matched 100% occupancy (Cell 8), so the gap is not an occupancy story. Most likely the `.local` accumulator spill flagged after Cell 5/6 (real per-MMA-iteration memory traffic a register-resident accumulator doesn't pay) -- occupancy parity did not translate into a speed win here, exactly the caution kernel-design.md rule 3 gives for chasing occupancy.
- **A real staging bug was found and fixed in this pass** (see the fixed `kernels/src/lib.rs` above): the shared-memory destination index for activation staging didn't account for which of the (up to 4) staging passes was running, so passes silently overwrote each other's data. Invisible at m_tiles=8 (only 1 pass) and at small ntok even for m_tiles=32 (later passes' write-gate suppressed the collision) -- only surfaced at ntok=256, where it corrupted the output completely (max_abs_diff=141.61). A real example of 'compiles and passes two cases' not meaning correct.
- **`gl_gemm_mma_q8_r256` (hand-written, NOT in production) failed correctness on ALL THREE cases** in this harness (max_abs_diff 15.74, 54.375, 136.05 -- growing with problem size, tolerance 5e-2). This harness uses synthetic, always-aligned buffers, so this is NOT the known CUDA_ERROR_MISALIGNED_ADDRESS crash (glcuda/src/runner.rs:146-165) -- it launched fine and computed wrong numbers. Per that same comment, r256's own parity test has never run on real hardware before this. Worth carrying back to the glcuda project independent of the cuda-oxide angle: r256 may have a genuine numerical bug in addition to the alignment crash.
- `gl_gemm_mma_q8_oxide` uses a runtime `m_tiles` parameter (not a const generic), so its register count at `m_tiles=8` (50, vs the hand kernel's exactly-sized 44) is a real, predicted-in-advance disadvantage -- confirmed small in Cell 7's numbers, but the `.local` traffic cost (not register count) looks like the bigger factor in the measured slowdown above. The natural follow-up is a `const M_TILES: u32` version (two monomorphized kernel instances, letting LLVM fully unroll and promote `acc` to registers) -- not built in this pass, flagged in `kernels/src/lib.rs`'s module doc as the Task 5 candidate most likely to close this gap.